In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_data/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/raw_data/Churn_Modelling.csv,Churn_Modelling.csv,684858,1790225727000
dbfs:/Volumes/workspace/default/raw_data/accounts.csv,accounts.csv,4763605,1791943473000
dbfs:/Volumes/workspace/default/raw_data/cards.csv,cards.csv,3980802,1791943472000
dbfs:/Volumes/workspace/default/raw_data/classification_report.png,classification_report.png,42515,1791607688000
dbfs:/Volumes/workspace/default/raw_data/confusion_matrix.png,confusion_matrix.png,37581,1791607687000
dbfs:/Volumes/workspace/default/raw_data/credit_risk_dataset.csv,credit_risk_dataset.csv,1804682,1791672097000
dbfs:/Volumes/workspace/default/raw_data/customers (1).csv,customers (1).csv,7741487,1791943473000
dbfs:/Volumes/workspace/default/raw_data/feature_importance.png,feature_importance.png,27725,1791607688000
dbfs:/Volumes/workspace/default/raw_data/loans.csv,loans.csv,1404977,1791943472000
dbfs:/Volumes/workspace/default/raw_data/synthetic_loan_approval_dataset.csv,synthetic_loan_approval_dataset.csv,4141493,1790642185000


In [0]:
base_path = "/Volumes/workspace/default/raw_data/"

# Read and save table
tables = {
    "customers": "customers.csv",
    "accounts": "accounts.csv",
    "transactions": "transactions.csv",
    "cards": "cards.csv",
    "loans": "loans.csv",
}
for table_name, file_name in tables.items():
    df = spark.read.csv(f"{base_path}{file_name}", header=True, inferSchema=True)
    df.write.mode("overwrite").saveAsTable(f"workspace.default.{table_name}")

# check
spark.sql("SELECT * FROM workspace.default.transactions LIMIT 5").show()


+--------------+----------+----------+------------+--------+----------+-----------------+
|transaction_id|account_id|  txn_date|    txn_type|  amount|   channel|merchant_category|
+--------------+----------+----------+------------+--------+----------+-----------------+
|       1109373|     36896|2023-03-30|Transfer Out| 3768.25|       POS|        Insurance|
|       1109374|     55207|2021-04-05|     Deposit| 6899.08|       POS|       Healthcare|
|       1109375|     79979|2021-02-03| Transfer In|14964.09|Mobile App|         Shopping|
|       1109376|     85310|2025-07-26|     Deposit| 3187.81|Mobile App|       Healthcare|
|       1109377|     89945|2020-05-14| Transfer In| 5553.39|Mobile App|    Fund Transfer|
+--------------+----------+----------+------------+--------+----------+-----------------+



In [0]:
%sql
-- Check Missing Value.
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS null_amount,
    SUM(CASE WHEN txn_date IS NULL THEN 1 ELSE 0 END) AS null_date
FROM workspace.default.transactions;


total_rows,null_amount,null_date
2000000,0,0


In [0]:
%sql
-- Check repeated id.
SELECT transaction_id, COUNT(*) AS cnt
FROM workspace.default.transactions
GROUP BY transaction_id
HAVING COUNT(*) > 1;


transaction_id,cnt


In [0]:
%sql
-- Check for negative or extremely large transaction amounts.
SELECT * FROM workspace.default.transactions
WHERE amount < 0 OR amount > 1000000;


transaction_id,account_id,txn_date,txn_type,amount,channel,merchant_category


In [0]:
%sql
-- Query for customer's income, credit score, account type, balance, and total transaction, sorted by total transaction amount in descending order.
SELECT 
    c.name AS customer_name,
    c.annual_income,
    c.credit_score,
    a.account_type,
    a.balance,
    SUM(t.amount) AS total_transaction_amount,
    COUNT(t.transaction_id) AS transaction_count
FROM workspace.default.transactions t
JOIN workspace.default.accounts a ON t.account_id = a.account_id
JOIN workspace.default.customers c ON a.customer_id = c.customer_id
GROUP BY c.name, c.annual_income, c.credit_score, a.account_type, a.balance
ORDER BY total_transaction_amount DESC


customer_name,annual_income,credit_score,account_type,balance,total_transaction_amount,transaction_count
John Brown,1908921,352,Salary,49081.88,355147.26,37
William Sharma,2967917,816,Fixed Deposit,35327.5,342321.16000000003,39
Karen Kumar,1686814,735,Savings,55920.18,335900.02,33
Anita Patel,2390606,325,Savings,34410.24,332855.65,29
William Chatterjee,2637782,754,Current,71718.64,328818.50999999995,25
Elizabeth Malhotra,2033429,553,Current,36047.74,324708.52999999997,37
Susan Garcia,1514816,457,Savings,38815.71,320580.17,37
John Malhotra,3302631,837,Current,58596.87,319135.26,38
Robert Reddy,851300,575,NRI,20903.62,316901.63,34
Jessica Martinez,2502355,898,Savings,5778.67,316541.04000000004,36


In [0]:
%sql
--Filter out the months with a total monthly transaction volume above the average
WITH MonthlyTotal AS (
    SELECT 
        DATE_TRUNC('month', txn_date) AS month,
        SUM(amount) AS total_amount,
        COUNT(*) AS txn_count
    FROM workspace.default.transactions
    GROUP BY 1
)
SELECT * FROM MonthlyTotal
WHERE total_amount > (SELECT AVG(total_amount) FROM MonthlyTotal)
ORDER BY month;


month,total_amount,txn_count
2018-01-01T00:00:00.000Z,1.1970587216000012E8,20003
2018-03-01T00:00:00.000Z,1.2102757088000017E8,19921
2018-05-01T00:00:00.000Z,1.2256830870000026E8,20162
2018-07-01T00:00:00.000Z,1.2141784803000014E8,20123
2018-08-01T00:00:00.000Z,1.2027254713000022E8,19895
2018-10-01T00:00:00.000Z,1.2094924950000001E8,19882
2019-01-01T00:00:00.000Z,1.2069137786999987E8,20043
2019-03-01T00:00:00.000Z,1.2015623325999989E8,19763
2019-05-01T00:00:00.000Z,1.1899882800000036E8,19741
2019-07-01T00:00:00.000Z,1.2187482620999976E8,19992


In [0]:
%sql
-- Cumulative Transaction Amount for Each Account (Sorted by Date)
SELECT 
    account_id,
    txn_date,
    amount,
    SUM(amount) OVER (
        PARTITION BY account_id 
        ORDER BY txn_date 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM workspace.default.transactions
ORDER BY account_id, txn_date
LIMIT 100;


account_id,txn_date,amount,running_total
1,2019-08-06,2632.22,2632.22
1,2019-10-17,12471.88,15104.099999999999
1,2020-06-30,11594.63,26698.729999999996
1,2021-02-05,1197.7,27896.429999999997
1,2021-03-23,2006.82,29903.249999999996
1,2021-04-23,4616.4,34519.649999999994
1,2021-06-11,3473.74,37993.38999999999
1,2021-09-14,8001.49,45994.87999999999
1,2022-04-14,10097.87,56092.74999999999
1,2022-06-06,8264.67,64357.41999999999


In [0]:
%sql
-- order by account_id, transaction_id
SELECT 
    account_id,
    transaction_id,
    amount,
    RANK() OVER (PARTITION BY account_id ORDER BY amount DESC) AS amount_rank
FROM workspace.default.transactions
LIMIT 100;


account_id,transaction_id,amount,amount_rank
12,1994239,13801.71,1
12,322735,13287.61,2
12,227574,9968.64,3
12,1433814,6534.66,4
12,1480395,6506.65,5
12,1243686,6150.94,6
12,1236257,4617.75,7
12,1413530,2948.81,8
12,1801064,2720.07,9
12,1097369,2530.97,10


In [0]:
%sql
-- Top 3 customers by transaction amount per channel
WITH ChannelCustomerRank AS (
    SELECT 
        t.channel,
        c.name AS customer_name,
        c.credit_score,
        SUM(t.amount) AS total_amount,
        RANK() OVER (PARTITION BY t.channel ORDER BY SUM(t.amount) DESC) AS rnk
    FROM workspace.default.transactions t
    JOIN workspace.default.accounts a ON t.account_id = a.account_id
    JOIN workspace.default.customers c ON a.customer_id = c.customer_id
    GROUP BY t.channel, c.name, c.credit_score
)
SELECT * FROM ChannelCustomerRank 
WHERE rnk <= 3 
ORDER BY channel, rnk;
